# 05 — Layer Normalization

**Ladder:** Multi-pass stats → fused per-row (Python) → masked power-of-two block → Triton forward (+ save $\mu$, $r_{\mathrm{std}}$).

**Study tip.** Fuse mean/var/affine so $x$ is not re-read from HBM between passes. Know how RMSNorm drops the mean. Match $\varepsilon$ when comparing to `F.layer_norm`.

Official: [Layer Norm](https://triton-lang.org/main/getting-started/tutorials/05-layer-norm.html)

Figures: [`docs/images/`](docs/images/) · guide: [`docs/visual-guide.md`](docs/visual-guide.md)

---

## 0. Definition

For a row $x \in \mathbb{R}^{N}$, weights $w,b \in \mathbb{R}^{N}$, and $\varepsilon > 0$:

$$
\mu = \frac{1}{N}\sum_{j=0}^{N-1} x_j,
\qquad
\sigma^{2} = \frac{1}{N}\sum_{j=0}^{N-1}(x_j-\mu)^{2},
\qquad
\hat{x}_j = \frac{x_j-\mu}{\sqrt{\sigma^{2}+\varepsilon}},
$$

$$
y_j = \hat{x}_j\, w_j + b_j.
$$

For $X \in \mathbb{R}^{M\times N}$, apply independently to each row.

**Affine equivariance (centering).** Adding $c\mathbf{1}$ to $x$ leaves $\hat{x}$ unchanged (mean shifts by $c$, differences cancel).

**Pass count.** Naive: separate reductions for $\mu$, $\sigma^{2}$, then normalize (multiple HBM sweeps). Fused: one program loads the row once, computes $(\mu,\sigma^{2},y)$ in SRAM.

![Whiteboard: multi-pass layer-norm vs fused row kernel](docs/images/benchmark-layer-norm.png)

*Theory board — mean/var/affine as separate HBM passes vs one fused load/reduce/store.*

### Theory board

![Whiteboard: multi-pass layer-norm vs fused row kernel](docs/images/whiteboard-theory.jpg)

*Theory board — mean/var/affine as separate HBM passes vs one fused load/reduce/store.*

### Pitfalls

- Two-pass mean then var can be less stable than Welford; still OK for teaching.
- Forgetting eps under the sqrt.
- Affine w,b must match the normalized feature axis length N.

### Self-check

1. Write μ, σ², x̂, y for a 4-vector by hand.
2. How many HBM reads of x in naive vs fused?
3. What changes for RMSNorm?


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import torch
import triton
import triton.language as tl

def _active_device():
    """Pick Triton active device, else CUDA, else CPU (study helper; no common/ package)."""
    try:
        import triton

        try:
            return triton.runtime.driver.active.get_active_torch_device()
        except Exception:
            pass
    except Exception:
        pass
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = _active_device()


def require_gpu() -> None:
    if not torch.cuda.is_available() and getattr(DEVICE, "type", None) != "cuda":
        pytest.skip(f"GPU not available (active device: {DEVICE})")



## 1–2. Naive multi-pass → fused row program

**Naive.** Three logical passes over $x$: mean, variance, affine normalize (`naive_layer_norm_pseudo`).

**Fused (still exact).** One loop body per row $r$ (`fused_layer_norm_pseudo`):

$$
\mu_r \;\rightarrow\; \sigma_r^{2} \;\rightarrow\; y_{r,:}
$$

with all temporaries local. Same numerics as the multi-pass form; fewer conceptual DRAM sweeps.

**Identity used in kernels.**

$$
\sigma^{2} = \frac{1}{N}\sum_j (x_j-\mu)^{2}
=
\Big(\frac{1}{N}\sum_j x_j^{2}\Big) - \mu^{2}.
$$

(optional one-pass Welford / two-sum variants exist; tutorials often use the centered form above in $\mathrm{fp32}$).

**Teaching sizes.** $M=4$, $N\in\{8,16,255\}$ — odd $N$ exercises the mask path before you scale to $1024+$.


In [ ]:
def naive_layer_norm_pseudo(x, weight, bias, eps=1e-5):
    """
    Separate passes over each row — easy to read, poor memory traffic.

    Pass 1: mean
    Pass 2: variance
    Pass 3: normalize + affine
    """
    M, N = x.shape
    y = torch.empty_like(x)
    for m in range(M):
        row = x[m]
        mean = row.sum() / N
        var = ((row - mean) ** 2).sum() / N
        y[m] = (row - mean) / torch.sqrt(var + eps) * weight + bias
    return y


In [ ]:
def fused_layer_norm_pseudo(x, weight, bias, eps=1e-5):
    """
    Still Python, but one function per row — mirrors the fused Triton program
    that owns a single row (pid = row index).
    """
    M, N = x.shape
    y = torch.empty_like(x)
    for m in range(M):
        row = x[m].to(torch.float32)
        mean = row.sum() / N
        centered = row - mean
        var = (centered * centered).sum() / N
        rstd = 1.0 / torch.sqrt(var + eps)
        y[m] = (centered * rstd * weight + bias).to(x.dtype)
    return y


## 3. Triton forward kernel

With $\mathrm{pid} = r$ (one row) and block size $B = \mathrm{next\_power\_of\_2}(N)$:

$$
\mu \leftarrow \tfrac{1}{N}\sum_j x_j,\quad
r_{\mathrm{std}} \leftarrow \big(\sigma^{2}+\varepsilon\big)^{-1/2},\quad
y_j \leftarrow (x_j-\mu)\, r_{\mathrm{std}}\, w_j + b_j.
$$

Stores also keep $\mu$ and $r_{\mathrm{std}}$ for a possible backward pass. Mask $j < N$ on a power-of-two block implements partial rows safely.

| Stage | Function | DRAM sweeps of $x$ |
|-------|----------|--------------------|
| Naive | `naive_layer_norm_pseudo` | multi-pass |
| Fused Python | `fused_layer_norm_pseudo` | fused locally |
| Triton | `_layer_norm_fwd_kernel` / `layer_norm` | one load + store $y$ |

**Cousin.** RMSNorm drops the mean: $y_j = w_j\, x_j / \sqrt{\tfrac{1}{N}\sum x^2+\varepsilon}$ — same fusion pattern, one fewer reduction.


In [ ]:
@triton.jit
def _layer_norm_fwd_kernel(
    Y,
    X,
    W,
    B,
    Mean,
    Rstd,
    stride,
    N,
    eps,
    BLOCK_SIZE: tl.constexpr,
):
    # One program per row — same ownership as fused_layer_norm_pseudo
    row = tl.program_id(0)
    Y += row * stride
    X += row * stride
    cols = tl.arange(0, BLOCK_SIZE)
    mask = cols < N
    x = tl.load(X + cols, mask=mask, other=0.0).to(tl.float32)
    mean = tl.sum(x, axis=0) / N
    x_centered = tl.where(mask, x - mean, 0.0)
    var = tl.sum(x_centered * x_centered, axis=0) / N
    rstd = 1 / tl.sqrt(var + eps)
    w = tl.load(W + cols, mask=mask, other=0.0).to(tl.float32)
    b = tl.load(B + cols, mask=mask, other=0.0).to(tl.float32)
    y = (x - mean) * rstd * w + b
    tl.store(Mean + row, mean)
    tl.store(Rstd + row, rstd)
    tl.store(Y + cols, y, mask=mask)


def layer_norm(x, weight, bias, eps=1e-5):
    M, N = x.shape
    y = torch.empty_like(x)
    mean = torch.empty((M,), device=x.device, dtype=torch.float32)
    rstd = torch.empty((M,), device=x.device, dtype=torch.float32)
    BLOCK_SIZE = triton.next_power_of_2(N)
    _layer_norm_fwd_kernel[(M,)](
        y,
        x,
        weight,
        bias,
        mean,
        rstd,
        x.stride(0),
        N,
        eps,
        BLOCK_SIZE=BLOCK_SIZE,
    )
    return y


In [ ]:
def test_pseudo_matches_torch() -> None:
    torch.manual_seed(0)
    x = torch.randn(4, 32)
    w = torch.randn(32)
    b = torch.randn(32)
    ref = torch.nn.functional.layer_norm(x, (32,), w, b)
    assert torch.allclose(naive_layer_norm_pseudo(x, w, b), ref, atol=1e-4, rtol=1e-4)
    assert torch.allclose(fused_layer_norm_pseudo(x, w, b), ref, atol=1e-4, rtol=1e-4)


def test_correctness() -> None:
    require_gpu()
    torch.manual_seed(0)
    x = torch.randn(128, 256, device=DEVICE)
    w = torch.randn(256, device=DEVICE)
    b = torch.randn(256, device=DEVICE)
    y = layer_norm(x, w, b)
    ref = torch.nn.functional.layer_norm(x, (256,), w, b)
    assert torch.allclose(y, ref, atol=1e-2, rtol=1e-2)
    print("layer_norm OK")
